In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, zipfile, shutil, platform, html, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid
try:
    from IPython.display import display, HTML, FileLink
except ImportError:
    display = print
    HTML = str
    FileLink = str

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/individual_swing_fingerprints'
FIG = ROOT / 'figures/individual_swing_fingerprints'
for folder in (RAW, OUT, FIG):
    folder.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
LOAD_OPTIONAL_OUTCOMES = True
LOOKBACK_S = 0.150
MIN_VALID_SWINGS = 5
SELECTED_HITTER_SESSION = None
if MIN_VALID_SWINGS < 4 or LOOKBACK_S <= 0:
    raise ValueError('Use at least four swings and a positive lookback duration.')
PROGRESS = np.linspace(0, 1, 101)
PHASES = ('approach', 'landing', 'post_plant')
CHANNELS = {'pelvis_angle_x': 'Pelvis tilt', 'torso_angle_x': 'Torso flexion/extension',
            'pelvis_angle_z': 'Pelvis axial rotation', 'torso_angle_z': 'Torso axial rotation'}
EVENTS = ('fp_10_time', 'fp_100_time', 'contact_time')
KEYS = ['hitter_id', 'session_id', 'hitter_side']
OUTCOMES = {'bat_speed_mph': 'bat_speed_mph_contact_x', 'exit_velocity_mph': 'exit_velo_mph_x'}
ARCHIVE_HASH = '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95'
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
ARCHIVE_URL = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/hitting_joint_angles.zip'
plt.style.use('seaborn-v0_8-whitegrid')
print('Study 16: Individual Swing Consistency Profiles')
print('Within-session movement descriptions; no mechanical quality score or causal claim.')
print('Pelvis X: posterior (+)/anterior (-) tilt. Torso X: extension (+)/flexion (-).')
print('Z angles retain OBP conventions; batting sides are analyzed separately.')
print('Primary cohort requires all three phases, valid chronology, and complete four-channel signals.')
print('Start-relative means subtracting each swing’s starting angle separately within each phase.')

In [ ]:
def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, target):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {target}; supply official data or enable AUTO_DOWNLOAD.')
    print('Downloading', target.name, flush=True)
    temp = target.with_suffix(target.suffix + '.part')
    with urlopen(url, timeout=120) as source, temp.open('wb') as dest:
        shutil.copyfileobj(source, dest)
    temp.replace(target)

target = RAW / 'joint_angles.csv'
if not target.exists():
    archive = RAW / 'hitting_joint_angles.zip'
    if not archive.exists():
        download(ARCHIVE_URL, archive)
    if file_hash(archive) != ARCHIVE_HASH:
        raise ValueError('Joint-angle archive checksum mismatch; remove and download again.')
    with zipfile.ZipFile(archive) as z:
        members = [n for n in z.namelist() if Path(n).name == 'joint_angles.csv']
        if len(members) != 1:
            raise ValueError('Expected exactly one joint_angles.csv in official archive.')
        temp = target.with_suffix('.csv.part')
        with z.open(members[0]) as source, temp.open('wb') as dest:
            shutil.copyfileobj(source, dest)
        temp.replace(target)
if not (RAW / 'metadata.csv').exists():
    download(CSV_BASE + 'metadata.csv', RAW / 'metadata.csv')
angles = pd.read_csv(target, usecols=['session_swing', 'time', *EVENTS, *CHANNELS],
                     dtype={'session_swing': str})
metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
required = ['session_swing', 'user', 'session', 'hitter_side']
if not set(required).issubset(metadata.columns):
    raise ValueError('Missing required metadata identity fields.')
if metadata.session_swing.isna().any() or metadata.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate metadata swing ID.')
if angles.session_swing.isna().any() or angles.duplicated(['session_swing', 'time']).any():
    raise ValueError('Missing angle swing IDs or duplicate swing/time keys.')
meta = metadata.set_index('session_swing')
groups = {str(k): g.sort_values('time').reset_index(drop=True)
          for k, g in angles.groupby('session_swing', sort=False)}
poi = pd.DataFrame(columns=['session_swing', *OUTCOMES.values()])
optional_outcome_status = 'disabled'
if LOAD_OPTIONAL_OUTCOMES:
    try:
        if not (RAW / 'poi_metrics.csv').exists():
            download(CSV_BASE + 'poi/poi_metrics.csv', RAW / 'poi_metrics.csv')
        poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
        optional_outcome_status = 'loaded'
    except OSError as exc:
        optional_outcome_status = str(exc)
        print('Optional outcomes unavailable; movement analysis continues:', exc)
if poi.session_swing.isna().any() or poi.session_swing.duplicated().any():
    raise ValueError('Missing or duplicate optional POI swing IDs.')
for col in OUTCOMES.values():
    if col not in poi:
        poi[col] = np.nan
print(f'{len(groups):,} angle trials; {metadata.user.nunique():,} hitters in metadata.')

In [ ]:
def get_events(g):
    ev = {}
    for name in EVENTS:
        x = g[name].to_numpy(float)
        if not np.isfinite(x).all() or np.ptp(x) > 1e-7:
            raise ValueError('missing_or_inconsistent_' + name)
        ev[name] = float(x[0])
    if not ev['fp_10_time'] < ev['fp_100_time'] < ev['contact_time']:
        raise ValueError('unordered_foot_contact_plant_ball_contact')
    return ev

def extract_swing(g, lookback=LOOKBACK_S):
    ev = get_events(g)
    boundaries = [ev['fp_10_time'] - lookback, ev['fp_10_time'],
                  ev['fp_100_time'], ev['contact_time']]
    t = g.time.to_numpy(float)
    if len(t) < 8 or not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid_time_grid')
    if t[0] > boundaries[0] + 1e-9 or t[-1] < boundaries[-1] - 1e-9:
        raise ValueError('incomplete_full_window_coverage')
    left = max(0, np.searchsorted(t, boundaries[0], side='right') - 1)
    right = min(len(t), np.searchsorted(t, boundaries[-1], side='left') + 1)
    t = t[left:right]
    dt = float(np.median(np.diff(t)))
    if not np.allclose(np.diff(t), dt, rtol=.03, atol=.00011):
        raise ValueError('sampling_gap_or_irregular_time_grid')
    signals = {}
    for ch in CHANNELS:
        a = g[ch].to_numpy(float)[left:right]
        if not np.isfinite(a).all():
            raise ValueError('nonfinite_' + ch)
        signals[ch] = np.rad2deg(np.unwrap(np.deg2rad(a)))
    curves, row = {}, {'sample_interval_ms': dt * 1000, **ev}
    for phase, start, end in zip(PHASES, boundaries[:-1], boundaries[1:]):
        if end - start < 7 * dt - 1e-9:
            raise ValueError('too_short_' + phase)
        row[phase + '_duration_ms'] = (end - start) * 1000
        for ch, a in signals.items():
            curves[(phase, ch)] = np.interp(start + PROGRESS * (end - start), t, a)
    return row, curves

def build_cohort(lookback=LOOKBACK_S):
    rows, audit, curves = [], [], {}
    for trial, g in groups.items():
        try:
            if trial not in meta.index:
                raise ValueError('missing_metadata_match')
            identity = meta.loc[trial]
            if identity[['user', 'session']].isna().any() or identity.hitter_side not in ('L', 'R'):
                raise ValueError('missing_identity_or_batting_side')
            row, trace = extract_swing(g, lookback)
            rows.append({'session_swing': trial, 'hitter_id': str(identity.user),
                         'session_id': str(identity.session), 'hitter_side': identity.hitter_side, **row})
            curves[trial] = trace
            reason = 'retained'
        except (ValueError, KeyError, TypeError) as exc:
            reason = str(exc)
        audit.append({'session_swing': trial, 'status': reason, 'lookback_ms': 1000 * lookback})
    features = pd.DataFrame(rows)
    if not features.empty:
        context = poi[['session_swing', *OUTCOMES.values()]].rename(columns={v:k for k,v in OUTCOMES.items()})
        features = features.merge(context, on='session_swing', how='left', validate='one_to_one')
    return features, pd.DataFrame(audit), curves

def branch_align(matrix):
    m = np.asarray(matrix, float).copy()
    if m.ndim != 2 or m.shape[1] != len(PROGRESS) or not np.isfinite(m).all():
        raise ValueError('invalid_curve_matrix')
    m -= 360 * np.round((m[:, [0]] - m[0, 0]) / 360)
    return m

def prepare_matrix(matrix, relative=False):
    m = branch_align(matrix)
    return m - m[:, [0]] if relative else m

def rms_curve(x):
    return float(np.sqrt(trapezoid(np.asarray(x, float) ** 2, PROGRESS)))

def curve_summary(m):
    if len(m) < 2:
        raise ValueError('at_least_two_swings_required')
    spread = m.std(axis=0, ddof=1)
    peak = int(np.argmax(spread))
    return {'rms_trajectory_sd_deg': rms_curve(spread),
            'peak_sd_deg': float(spread[peak]), 'peak_sd_progress_pct': float(100 * PROGRESS[peak]),
            'start_angle_sd_deg': float(spread[0]), 'end_angle_sd_deg': float(spread[-1])}

def summarize(features, curves, minimum=MIN_VALID_SWINGS, diagnostics=True):
    metrics, timing, deviations, influence = [], [], [], []
    if features.empty:
        return (pd.DataFrame(),) * 4
    for key, g in features.groupby(KEYS, sort=True):
        if len(g) < minimum:
            continue
        identity = dict(zip(KEYS, key))
        ids = g.session_swing.tolist()
        for phase in PHASES:
            duration = g[phase + '_duration_ms'].to_numpy(float)
            timing.append({**identity, 'phase': phase, 'n_swings': len(g),
                           'mean_duration_ms': duration.mean(), 'sd_duration_ms': duration.std(ddof=1),
                           'min_duration_ms': duration.min(), 'max_duration_ms': duration.max()})
            for ch in CHANNELS:
                raw = np.stack([curves[k][(phase, ch)] for k in ids])
                for relative in (False, True):
                    m = prepare_matrix(raw, relative)
                    mode = 'start_relative' if relative else 'absolute'
                    baseline = curve_summary(m)
                    base = {**identity, 'phase': phase, 'channel': ch, 'mode': mode,
                            'n_swings': len(g)}
                    metrics.append({**base, **baseline})
                    if not diagnostics:
                        continue
                    other_means = (m.sum(axis=0) - m) / (len(m) - 1)
                    for i, trial in enumerate(ids):
                        score = rms_curve(m[i] - other_means[i])
                        context = g.iloc[i]
                        deviations.append({**base, 'session_swing': trial, 'loo_rms_distance_deg': score,
                                           **{outcome: context[outcome] for outcome in OUTCOMES}})
                        omitted = curve_summary(np.delete(m, i, axis=0))
                        influence.append({**base, 'omitted_swing': trial, 'remaining_n': len(m) - 1,
                                          'remaining_meets_minimum': len(m) - 1 >= minimum,
                                          'baseline_rms_sd_deg': baseline['rms_trajectory_sd_deg'],
                                          'omission_rms_sd_deg': omitted['rms_trajectory_sd_deg'],
                                          'change_rms_sd_deg': omitted['rms_trajectory_sd_deg'] - baseline['rms_trajectory_sd_deg']})
    return tuple(pd.DataFrame(x) for x in (metrics, timing, deviations, influence))

In [ ]:
def run_measurement_checks():
    m = np.stack([PROGRESS * 10, PROGRESS * 10 + 2, PROGRESS * 10 + 4])
    assert np.isclose(curve_summary(prepare_matrix(m))['rms_trajectory_sd_deg'], 2)
    assert curve_summary(prepare_matrix(m, True))['rms_trajectory_sd_deg'] < 1e-12
    wrapped = m.copy(); wrapped[1] += 360
    np.testing.assert_allclose(prepare_matrix(m), prepare_matrix(wrapped), atol=1e-10)
    others = (m.sum(axis=0) - m) / (len(m) - 1)
    for i in range(3):
        np.testing.assert_allclose(others[i], np.delete(m, i, axis=0).mean(axis=0))
    assert np.isclose(rms_curve(PROGRESS), np.sqrt(trapezoid(PROGRESS ** 2, PROGRESS)))
    t = np.arange(-.3, .31, 1/360)
    g = pd.DataFrame({'time': t, 'fp_10_time': 0., 'fp_100_time': .05, 'contact_time': .18})
    for ch in CHANNELS:
        g[ch] = t * 50 + 20
    row, trace = extract_swing(g)
    assert np.isclose(row['landing_duration_ms'], 50)
    np.testing.assert_allclose(trace[('post_plant', 'pelvis_angle_x')][[0, -1]], [22.5, 29.])
    b = g.copy(); b['pelvis_angle_z'] = ((170 + t * 100 + 180) % 360) - 180
    unwrapped = extract_swing(b)[1][('post_plant','pelvis_angle_z')]
    np.testing.assert_allclose(unwrapped[[0,-1]], [175,188], atol=1e-10)
    failures = []
    b = g.copy(); b['fp_10_time'] = .2; failures.append(b)
    b = g.copy(); b.loc[10, 'contact_time'] = .19; failures.append(b)
    b = g.copy(); b.loc[b.time.abs().idxmin(), 'torso_angle_x'] = np.nan; failures.append(b)
    b = g.drop(g.time.abs().idxmin()); failures.append(b)
    b = g.copy(); b['fp_100_time'] = .01; failures.append(b)
    failures.append(g.loc[g.time > -.1])
    for bad in failures:
        try:
            extract_swing(bad)
        except ValueError:
            pass
        else:
            raise AssertionError('Invalid trial passed eligibility.')
    print('Measurement checks passed: phase boundaries, event order, short windows, gaps, missing signals,')
    print('branch alignment, offset-versus-shape separation, sample SD, and leave-one-out reference curves.')

run_measurement_checks()
features, audit, curves = build_cohort()
audit.to_csv(OUT / 'trial_eligibility_audit.csv', index=False)
if features.empty:
    raise ValueError('No complete swings passed eligibility; inspect trial_eligibility_audit.csv.')
metrics, timing, deviations, influence = summarize(features, curves)
if metrics.empty:
    raise ValueError('No session has enough complete swings; inspect audit or set minimum to 4 or more.')
if MIN_VALID_SWINGS < 4:
    raise ValueError('Use at least four swings for reports and omission diagnostics.')
session_counts = features.groupby(KEYS).size().rename('n_swings').reset_index()
session_counts['report_eligible'] = session_counts.n_swings >= MIN_VALID_SWINGS
display(audit.status.value_counts().rename_axis('status').to_frame('trials'))
print(f'{len(features)} complete swings; {metrics[KEYS].drop_duplicates().shape[0]} eligible hitter-session-side profiles.')
display(timing.head(9).round(3))

In [ ]:
sensitivity_rows = []
for minimum in (4, 5, 6):
    table = summarize(features, curves, minimum=minimum, diagnostics=False)[0]
    for _, r in table.iterrows():
        sensitivity_rows.append({'sensitivity': 'minimum_swings', 'setting': minimum,
                                 **r.to_dict()})
lookback_audits, lookback_counts = [], []
lookback_data = {}
for lookback in (.100, .150, .200):
    f, a, c = (features, audit, curves) if lookback == LOOKBACK_S else build_cohort(lookback)
    lookback_data[lookback] = (f, c)
    lookback_audits.append(a)
    lookback_counts.append({'lookback_ms': lookback * 1000, 'complete_swings': len(f)})
common_ids = set.intersection(*(set(f.session_swing) for f, _ in lookback_data.values()))
for lookback, (f, c) in lookback_data.items():
    common = f.loc[f.session_swing.isin(common_ids)]
    table = summarize(common, c, diagnostics=False)[0]
    for _, r in table.iterrows():
        sensitivity_rows.append({'sensitivity': 'lookback_common_swings', 'setting': 1000 * lookback,
                                 **r.to_dict()})
sensitivities = pd.DataFrame(sensitivity_rows)
pd.concat(lookback_audits, ignore_index=True).to_csv(OUT / 'lookback_trial_audits.csv', index=False)
pd.DataFrame(lookback_counts).to_csv(OUT / 'lookback_cohort_counts.csv', index=False)
paired_rows = []
join = [*KEYS, 'phase', 'channel', 'mode']
for setting in (100., 200.):
    sub = sensitivities.loc[sensitivities.sensitivity.eq('lookback_common_swings')]
    baseline = sub.loc[sub.setting.eq(150.), join + ['rms_trajectory_sd_deg']]
    alternate = sub.loc[sub.setting.eq(setting), join + ['rms_trajectory_sd_deg']]
    pair = baseline.merge(alternate, on=join, suffixes=('_150ms', '_alternate'), validate='one_to_one')
    pair['setting_ms'] = setting
    pair['change_deg'] = pair.rms_trajectory_sd_deg_alternate - pair.rms_trajectory_sd_deg_150ms
    paired_rows.append(pair)
paired = pd.concat(paired_rows, ignore_index=True)
print('Common-cohort lookback sensitivity:', len(common_ids), 'swings.')
print('Landing/post-plant comparisons should be unchanged; approach measures use different windows.')
if not paired.empty:
    unaffected = paired.loc[paired.phase.ne('approach'), 'change_deg']
    assert np.allclose(unaffected, 0, atol=1e-10)
display(pd.DataFrame(lookback_counts))

In [ ]:
from scipy.stats import spearmanr

DURATION_OUT = OUT / 'phase_duration_audit'
DURATION_FIG = FIG / 'phase_duration_audit'
for folder in (DURATION_OUT, DURATION_FIG):
    folder.mkdir(parents=True, exist_ok=True)
DURATION_FENCE_MULTIPLIER = 3.0
FOCUS_DURATION_SESSION = ('000009', '000123', 'R')
duration_audit = features[['session_swing', *KEYS, *[p+'_duration_ms' for p in PHASES]]].copy()
duration_distribution_rows = []
for phase in PHASES:
    values = features[phase+'_duration_ms']
    q1, median, q3 = values.quantile([.25,.5,.75])
    fence = q3 + DURATION_FENCE_MULTIPLIER * (q3-q1)
    flag = (values > fence + 1e-7) if phase != 'approach' else pd.Series(False, index=values.index)
    duration_audit[phase+'_upper_fence_ms'] = fence
    duration_audit[phase+'_duration_review_flag'] = flag
    duration_distribution_rows.append({'phase':phase, 'n_swings':len(values),
        'min_ms':values.min(), 'q1_ms':q1, 'median_ms':median, 'q3_ms':q3,
        'p95_ms':values.quantile(.95), 'p99_ms':values.quantile(.99), 'max_ms':values.max(),
        'upper_fence_ms':fence, 'n_review_flags':int(flag.sum()),
        'rule':'fixed window; not screened' if phase=='approach' else 'duration > pooled Q3 + 3*IQR'})
duration_distribution = pd.DataFrame(duration_distribution_rows)
duration_audit['any_duration_review_flag'] = duration_audit[[p+'_duration_review_flag' for p in PHASES]].any(axis=1)
duration_session_rows = []
for key, g in duration_audit.groupby(KEYS):
    identity = dict(zip(KEYS,key))
    for phase in PHASES:
        v = g[phase+'_duration_ms']
        duration_session_rows.append({**identity, 'phase':phase, 'n_swings':len(g),
            'primary_report_eligible':len(g)>=MIN_VALID_SWINGS,
            'mean_ms':v.mean(), 'sd_ms':v.std(ddof=1), 'min_ms':v.min(), 'max_ms':v.max(),
            'upper_fence_ms':g[phase+'_upper_fence_ms'].iloc[0],
            'review_flags':int(g[phase+'_duration_review_flag'].sum())})
duration_sessions = pd.DataFrame(duration_session_rows)

duration_rules = {
    'robust_upper_fence':(~duration_audit.any_duration_review_flag).to_numpy(),
    'landing_max_200ms':features.landing_duration_ms.le(200+1e-7).to_numpy(),
    'landing_max_300ms':features.landing_duration_ms.le(300+1e-7).to_numpy(),
    'landing_max_400ms':features.landing_duration_ms.le(400+1e-7).to_numpy(),
    'post_plant_max_200ms':features.post_plant_duration_ms.le(200+1e-7).to_numpy()}
duration_rule_descriptions = {
    'robust_upper_fence':'Omit any swing above the frozen pooled Q3 + 3*IQR fence in landing or post-plant.',
    'landing_max_200ms':'Omit swings with landing duration > 200 ms.',
    'landing_max_300ms':'Omit swings with landing duration > 300 ms.',
    'landing_max_400ms':'Omit swings with landing duration > 400 ms.',
    'post_plant_max_200ms':'Omit swings with post-plant duration > 200 ms.'}
comparison_keys = [*KEYS, 'phase', 'channel', 'mode']
duration_scenarios, duration_attrition, duration_pairs, duration_screened = [], [], [], []
duration_membership = duration_audit[['session_swing',*KEYS]].copy()
primary_counts = features.groupby(KEYS).size().rename('primary_n_swings')
for scenario, keep in duration_rules.items():
    retained = features.loc[keep].copy()
    table = summarize(retained, curves, minimum=MIN_VALID_SWINGS, diagnostics=False)[0]
    if table.empty:
        table = pd.DataFrame(columns=metrics.columns)
    duration_screened.append(table.assign(scenario=scenario))
    duration_membership[scenario+'_retained'] = keep
    counts = primary_counts.to_frame().join(retained.groupby(KEYS).size().rename('screened_n_swings'), how='left').fillna(0).reset_index()
    counts['screened_n_swings'] = counts.screened_n_swings.astype(int)
    counts['primary_eligible'] = counts.primary_n_swings >= MIN_VALID_SWINGS
    counts['screened_eligible'] = counts.screened_n_swings >= MIN_VALID_SWINGS
    counts['lost_primary_profile'] = counts.primary_eligible & ~counts.screened_eligible
    counts['scenario'] = scenario
    duration_attrition.append(counts)
    duration_scenarios.append({'scenario':scenario, 'rule':duration_rule_descriptions[scenario],
        'retained_swings':len(retained), 'omitted_swings':int((~keep).sum()),
        'primary_profiles':int(counts.primary_eligible.sum()),
        'screened_profiles':int(counts.screened_eligible.sum()),
        'lost_primary_profiles':int(counts.lost_primary_profile.sum())})
    measurements = ['n_swings','rms_trajectory_sd_deg','peak_sd_deg','peak_sd_progress_pct',
                    'start_angle_sd_deg','end_angle_sd_deg']
    pair = metrics[comparison_keys+measurements].merge(table[comparison_keys+measurements],
        on=comparison_keys, suffixes=('_primary','_screened'), validate='one_to_one')
    pair['scenario'] = scenario
    pair['change_rms_sd_deg'] = pair.rms_trajectory_sd_deg_screened-pair.rms_trajectory_sd_deg_primary
    denominator = pair.rms_trajectory_sd_deg_primary.where(pair.rms_trajectory_sd_deg_primary.abs()>1e-9)
    pair['pct_change_rms_sd'] = 100*pair.change_rms_sd_deg/denominator
    duration_pairs.append(pair)
    if keep.all():
        assert np.allclose(pair.change_rms_sd_deg,0,atol=1e-10)
    assert set(retained.session_swing) <= set(features.session_swing)
    assert not counts.loc[counts.primary_eligible.eq(False),'screened_eligible'].any()
duration_scenarios = pd.DataFrame(duration_scenarios)
duration_attrition = pd.concat(duration_attrition,ignore_index=True)
duration_pairs = pd.concat(duration_pairs,ignore_index=True)
duration_screened = pd.concat(duration_screened,ignore_index=True)
duration_summary_rows = []
for key,g in duration_pairs.groupby(['scenario','phase','channel','mode']):
    x,y = g.rms_trajectory_sd_deg_primary,g.rms_trajectory_sd_deg_screened
    rho = float(spearmanr(x,y).statistic) if len(g)>=3 and x.nunique()>1 and y.nunique()>1 else np.nan
    duration_summary_rows.append(dict(zip(['scenario','phase','channel','mode'],key)) | {
        'paired_profiles':len(g), 'median_primary_sd_deg':x.median(),'median_screened_sd_deg':y.median(),
        'median_paired_change_deg':g.change_rms_sd_deg.median(),
        'median_absolute_change_deg':g.change_rms_sd_deg.abs().median(),
        'p90_absolute_change_deg':g.change_rms_sd_deg.abs().quantile(.9),
        'max_absolute_change_deg':g.change_rms_sd_deg.abs().max(), 'rank_rho':rho})
duration_summary = pd.DataFrame(duration_summary_rows)
for name,table in [('trial_duration_audit',duration_audit),('duration_distribution',duration_distribution),
    ('session_duration_audit',duration_sessions),('scenario_cohort_counts',duration_scenarios),
    ('scenario_session_attrition',duration_attrition),('scenario_trial_membership',duration_membership),
    ('screened_curve_profiles',duration_screened),('paired_profile_changes',duration_pairs),
    ('paired_change_summary',duration_summary)]:
    table.to_csv(DURATION_OUT/(name+'.csv'),index=False)

fig, axes = plt.subplots(1,2,figsize=(13,4.5))
for ax,phase in zip(axes,['landing','post_plant']):
    values = features[phase+'_duration_ms']
    ax.hist(values,bins=40,color='#167D9A',alpha=.8)
    fence = duration_distribution.set_index('phase').loc[phase,'upper_fence_ms']
    ax.axvline(fence,color='#A83C51',ls='--',label=f'Q3 + 3 IQR: {fence:.1f} ms')
    ax.set(title=phase.replace('_',' ').title(),xlabel='Recorded duration (ms)',ylabel='Swings')
    ax.legend(fontsize=9)
fig.suptitle('Phase-duration audit | review flags, not invalid-event labels')
fig.tight_layout()
fig.savefig(DURATION_FIG/'duration_distributions.png',dpi=140,bbox_inches='tight')
plt.show(); plt.close(fig)

focus_mask = np.logical_and.reduce([features[c].eq(v).to_numpy() for c,v in zip(KEYS,FOCUS_DURATION_SESSION)])
duration_focus = features.loc[focus_mask].copy()
duration_focus.to_csv(DURATION_OUT/'focus_trial_durations.csv',index=False)
focus_attrition = duration_attrition.loc[np.logical_and.reduce([
    duration_attrition[c].eq(v).to_numpy() for c,v in zip(KEYS,FOCUS_DURATION_SESSION)])]
focus_attrition.to_csv(DURATION_OUT/'focus_scenario_eligibility.csv',index=False)
if not duration_focus.empty:
    fig,axes = plt.subplots(2,2,figsize=(14,9))
    palette = plt.get_cmap('tab10')
    for ax,ch in zip(axes.flat,CHANNELS):
        for j,(_,r) in enumerate(duration_focus.iterrows()):
            start_ms = -(LOOKBACK_S*1000+r.landing_duration_ms+r.post_plant_duration_ms)
            boundaries = [start_ms,-r.landing_duration_ms-r.post_plant_duration_ms,-r.post_plant_duration_ms,0.]
            times,angles_real = [],[]
            for k,phase in enumerate(PHASES):
                t = np.linspace(boundaries[k],boundaries[k+1],len(PROGRESS))
                a = curves[r.session_swing][(phase,ch)]
                times.extend(t if k==0 else t[1:]); angles_real.extend(a if k==0 else a[1:])
            color = palette(j%10)
            ax.plot(times,angles_real,color=color,lw=1.3,label=r.session_swing)
            for pos in boundaries[1:3]:
                ax.scatter(pos,np.interp(pos,times,angles_real),color=color,s=22)
        ax.axvline(0,color='black',ls=':',lw=1)
        ax.set(title=CHANNELS[ch],xlabel='Time relative to ball contact (ms)',ylabel='Angle (deg)')
    axes[0,0].legend(fontsize=8,ncol=3)
    fig.suptitle('Duration review: '+' / '.join(FOCUS_DURATION_SESSION)+' | dots: foot contact and full plant')
    fig.tight_layout(rect=(0,0,1,.96))
    fig.savefig(DURATION_FIG/'focus_real_time_curves.png',dpi=140,bbox_inches='tight')
    plt.show();plt.close(fig)

duration_audit_manifest = {'primary_preserved':True,'annotations_repaired':False,
    'review_fence':'pooled complete-swing Q3 + 3*IQR; frozen before screening',
    'fixed_thresholds':'analytical sensitivity settings, not validated physiological bounds',
    'rules':duration_rule_descriptions,'minimum_swings':MIN_VALID_SWINGS,
    'comparison':'same hitter-session-side pairs, primary all swings versus screened swings; lost profiles exported separately',
    'shared_cohort':'a screened trial ID is omitted from every phase, channel and mode',
    'inference':'descriptive changes and rank agreement; no p-values or test-retest claim'}
(DURATION_OUT/'audit_settings.json').write_text(json.dumps(duration_audit_manifest,indent=2),encoding='utf-8')
print('Duration flags identify review candidates; no annotation is repaired or primary trial removed.')
print('Fixed caps are sensitivity settings, not established physiological validity limits.')
display(duration_distribution.round(3))
display(duration_scenarios)
print('Focused session eligibility after screening:')
display(focus_attrition)
print('Largest paired changes; sessions falling below the minimum are in the attrition table, not these pairs:')
display(duration_pairs.loc[duration_pairs.change_rms_sd_deg.abs().nlargest(10).index,
    ['scenario',*KEYS,'phase','channel','mode','n_swings_primary','n_swings_screened','change_rms_sd_deg']].round(3))


In [ ]:
def selected_rows(table, key):
    mask = np.ones(len(table), dtype=bool)
    for col, value in zip(KEYS, key):
        mask &= table[col].eq(value).to_numpy()
    return table.loc[mask]

eligible_keys = [tuple(r) for r in metrics[KEYS].drop_duplicates().itertuples(index=False, name=None)]
if SELECTED_HITTER_SESSION is None:
    selected_key = sorted(eligible_keys)[0]
else:
    selected_key = tuple(str(x) for x in SELECTED_HITTER_SESSION)
    if selected_key not in eligible_keys:
        raise ValueError('SELECTED_HITTER_SESSION must be an eligible (hitter_id, session_id, hitter_side) tuple.')

def plot_profile(key, relative=False, axial=False, save=True):
    g = selected_rows(features, key)
    channels = [c for c in CHANNELS if c.endswith('_z' if axial else '_x')]
    fig, axes = plt.subplots(4, 3, figsize=(16, 13), gridspec_kw={'height_ratios': [3,1,3,1]})
    for segment, ch in enumerate(channels):
        color = '#167D9A' if ch.startswith('pelvis') else '#A83C51'
        for j, phase in enumerate(PHASES):
            m = prepare_matrix(np.stack([curves[k][(phase,ch)] for k in g.session_swing]), relative)
            mean, sd = m.mean(axis=0), m.std(axis=0, ddof=1)
            ax, spread = axes[2*segment,j], axes[2*segment+1,j]
            ax.fill_between(PROGRESS*100, mean-sd, mean+sd, color=color, alpha=.20, label='Across-swing ±1 SD')
            ax.plot(PROGRESS*100, m.T, color=color, alpha=.22, lw=.8)
            ax.plot(PROGRESS*100, mean, color=color, lw=2.5, label='Session mean')
            ax.set_ylabel(CHANNELS[ch] + ' (deg)')
            duration = g[phase+'_duration_ms']
            ax.set_title(f'{phase.replace("_"," ").title()} | {duration.mean():.1f} ± {duration.std(ddof=1):.1f} ms')
            ax.set_xlim(0,100)
            if segment == 0 and j == 0:
                ax.legend(fontsize=8)
            spread.plot(PROGRESS*100,sd,color=color,lw=1.7)
            spread.set(xlim=(0,100), ylim=(0,max(sd.max()*1.1,.05)), ylabel='SD (deg)', xlabel='Phase progress (%)')
    mode = 'start_relative' if relative else 'absolute'
    kind = 'axial' if axial else 'posture'
    fig.suptitle(f'Hitter {key[0]} | Session {key[1]} | {key[2]} | n={len(g)} | {mode.replace("_"," ")}', fontsize=15)
    fig.tight_layout(rect=(0,0,1,.97))
    if save:
        fig.savefig(FIG / (f'{"_".join(key)}_{kind}_{mode}.png'), dpi=130, bbox_inches='tight')
    plt.show()
    plt.close(fig)

for relative in (False, True):
    for axial in (False, True):
        plot_profile(selected_key, relative, axial)
display(selected_rows(metrics, selected_key).round(3))
print('Curve bands are across-swing sample SD, not confidence or normal-mechanics intervals.')
print('Individual curves are aligned within phase; phase-duration SD retains timing variation.')

In [ ]:
payload = []
for key in sorted(eligible_keys):
    g = selected_rows(features, key)
    ids = g.session_swing.tolist()
    traces = {}
    for phase in PHASES:
        for ch in CHANNELS:
            traces[phase + '|' + ch] = branch_align(np.stack([curves[k][(phase,ch)] for k in ids])).round(5).tolist()
    summary = selected_rows(metrics, key).copy()
    time_summary = selected_rows(timing, key).copy()
    dev = selected_rows(deviations, key).copy()
    infl = selected_rows(influence, key).copy()
    payload.append({'key': list(key), 'ids': ids, 'curves': traces,
                    'summary': json.loads(summary.to_json(orient='records')),
                    'timing': json.loads(time_summary.to_json(orient='records')),
                    'deviations': json.loads(dev.to_json(orient='records')),
                    'influence': json.loads(infl.to_json(orient='records')),
                    'duration_review': json.loads(selected_rows(duration_sessions,key).to_json(orient='records'))})
report_data = {'sessions': payload, 'phases': list(PHASES), 'channels': CHANNELS,
               'minimum': MIN_VALID_SWINGS}
report_json = json.dumps(report_data, allow_nan=False, separators=(',', ':')).replace('</', '<' + chr(92) + '/')

In [ ]:
REPORT_TEMPLATE = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Individual Swing Consistency Profiles</title>\n<style>\n:root{font-family:system-ui,sans-serif;color:#e6edf4;background:#0c1520;font-size:14px}body{max-width:1480px;margin:auto;padding:28px}h1{font-size:29px;margin-bottom:8px}h2{font-size:20px;margin-top:30px}p{color:#b3c2d1;line-height:1.6}.controls{display:flex;gap:18px;flex-wrap:wrap;padding:18px;background:#162434;border-radius:10px;position:sticky;top:0;z-index:5}label{display:flex;gap:8px;align-items:center}select{padding:9px;background:#0c1520;color:#edf3f9;border:1px solid #496175;border-radius:5px;max-width:360px}.grid{display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px}.panel{background:#162434;border:1px solid #2b4053;border-radius:9px;padding:12px;min-width:0}.panel h3{font-size:14px;margin:0 0 3px}.small{font-size:12px;color:#a4b7c8}canvas{display:block;width:100%;height:auto}table{border-collapse:collapse;width:100%;font-size:12px}td,th{text-align:left;padding:9px;border-bottom:1px solid #2b4053}th{color:#a4c4db}tbody tr:hover{background:#20364b}.scroll{overflow-x:auto}.legend{display:flex;gap:22px;margin:14px 0;color:#b3c2d1}.pill{display:inline-block;background:#193849;border-radius:5px;padding:7px;margin:4px 12px 4px 0}.note{background:#162434;border-left:3px solid #4b91af;padding:12px 16px}.footer{margin-top:28px;font-size:12px}@media(max-width:1000px){.grid{grid-template-columns:repeat(2,minmax(0,1fr))}}@media(max-width:650px){body{padding:14px}.grid{grid-template-columns:1fr}.controls{position:static}}\n</style></head><body>\n<h1>Individual Swing Consistency Profiles</h1><p>Pelvis and torso movement across repeated swings. OpenBiomechanics Dataset v1 · Study 16</p>\n<div class="controls"><label>Hitter / session / side <select id="session"></select></label><label>Angle view <select id="mode"><option value="absolute">Absolute angles</option><option value="start_relative">Start-relative angles</option></select></label><label>Highlight swing <select id="swing"></select></label></div>\n<p class="note">These profiles describe observed movement within one session. Greater spread does not establish worse mechanics. Start-relative curves subtract each swing’s starting angle separately within each phase. Swing highlighting changes the display only; all retained reps remain in the calculations.</p>\n<div id="counts"></div><div id="duration-review"></div><div id="timing"></div>\n<div class="legend"><span>Thin lines: individual swings</span><span>Bold line: session mean</span><span>Shading: across-swing ±1 SD</span><span style="color:#ffd16a">Gold: selected swing</span></div>\n<p class="small">Each phase is normalized separately. Bottom strip: pointwise across-swing SD. Bands are sample SD, not confidence intervals or reference ranges. Pelvis X = posterior (+) / anterior (−) tilt; torso X = extension (+) / flexion (−). Z angles retain source conventions.</p>\n<div id="plots" class="grid"></div>\n<h2>Curve summary</h2><div id="summary" class="scroll"></div>\n<h2>Reps to review</h2><p>Largest distances from the mean of the other swings, within the selected session and angle view. Repeated swing IDs represent different phases or channels. Rankings are descriptive; no rep is automatically excluded. Bat speed and exit velocity are optional context.</p><div id="deviations" class="scroll"></div>\n<h2>Omission sensitivity</h2><p>Largest absolute changes in trajectory SD when one rep is omitted. Negative changes mean the band narrows. A diagnostic may have fewer reps than the reporting minimum; the remaining count is shown.</p><div id="influence" class="scroll"></div>\n<p class="footer">“Fingerprint” is a descriptive profile, not a validated persistent identity. Normalized curves hide timing differences, so phase durations are reported separately. The approach is a fixed pre-foot-contact window, not measured stride onset. Single-view iPhone CV does not directly measure these 3D joint angles; a separate validation study is required.<br>Source: <a href="https://github.com/drivelineresearch/openbiomechanics" style="color:#82c5e4">Driveline OpenBiomechanics</a> · No external scripts or services are required to view this report.</p>\n<script id="report-data" type="application/json">__REPORT_DATA__</script>\n<script>\nconst data=JSON.parse(document.getElementById(\'report-data\').textContent),$=id=>document.getElementById(id);\nconst esc=x=>String(x).replace(/[&<>"\']/g,c=>({\'&\':\'&amp;\',\'<\':\'&lt;\',\'>\':\'&gt;\',\'"\':\'&quot;\',"\'":\'&#39;\'}[c]));\nconst fmt=(v,n=2)=>v===null||v===undefined||!Number.isFinite(Number(v))?\'—\':Number(v).toFixed(n);\nconst title=p=>({approach:\'Approach → foot contact\',landing:\'Foot contact → full plant\',post_plant:\'Full plant → ball contact\'}[p]);\nconst table=(headers,rows)=>\'<table><thead><tr>\'+headers.map(x=>\'<th>\'+esc(x)+\'</th>\').join(\'\')+\'</tr></thead><tbody>\'+rows.map(row=>\'<tr>\'+row.map(x=>\'<td>\'+esc(x)+\'</td>\').join(\'\')+\'</tr>\').join(\'\')+\'</tbody></table>\';\ndata.sessions.forEach((s,i)=>{$(\'session\').add(new Option(s.key.join(\' / \'),i))});\nfunction line(ctx,values,x,y,color,width=1){ctx.beginPath();values.forEach((v,i)=>{let px=x(i),py=y(v);i?ctx.lineTo(px,py):ctx.moveTo(px,py)});ctx.strokeStyle=color;ctx.lineWidth=width;ctx.stroke()}\nfunction plot(canvas,m,color,highlight){\n const ctx=canvas.getContext(\'2d\'),W=640,H=390;canvas.width=W;canvas.height=H;ctx.clearRect(0,0,W,H);\n const mean=m[0].map((_,i)=>m.reduce((a,r)=>a+r[i],0)/m.length),sd=mean.map((mu,i)=>Math.sqrt(m.reduce((a,r)=>a+(r[i]-mu)**2,0)/(m.length-1)));\n let lo=Math.min(...m.flat(),...mean.map((v,i)=>v-sd[i])),hi=Math.max(...m.flat(),...mean.map((v,i)=>v+sd[i]));let pad=Math.max((hi-lo)*.09,.15);lo-=pad;hi+=pad;\n const left=68,right=W-18,top=20,bottom=250,x=i=>left+(right-left)*i/100,y=v=>bottom-(bottom-top)*(v-lo)/(hi-lo),sy=v=>345-65*v/Math.max(...sd,.05);\n ctx.font=\'16px system-ui\';ctx.fillStyle=\'#9bb2c5\';ctx.strokeStyle=\'#30485c\';ctx.lineWidth=1;\n for(let k=0;k<=4;k++){let value=lo+(hi-lo)*k/4,yy=y(value);ctx.beginPath();ctx.moveTo(left,yy);ctx.lineTo(right,yy);ctx.stroke();ctx.fillText(value.toFixed(1),4,yy+5)}\n ctx.beginPath();mean.forEach((v,i)=>i?ctx.lineTo(x(i),y(v+sd[i])):ctx.moveTo(x(i),y(v+sd[i])));for(let i=100;i>=0;i--)ctx.lineTo(x(i),y(mean[i]-sd[i]));ctx.closePath();ctx.fillStyle=color+\'35\';ctx.fill();\n m.forEach(row=>line(ctx,row,x,y,color+\'66\',1.2));line(ctx,mean,x,y,color,3.6);if(highlight>=0)line(ctx,m[highlight],x,y,\'#ffd16a\',3);\n ctx.fillStyle=\'#9bb2c5\';ctx.fillText(\'deg\',7,17);ctx.fillText(\'SD\',9,292);line(ctx,sd,x,sy,color,2.5);\n ctx.fillText(\'0\',25,348);ctx.fillText(Math.max(...sd,.05).toFixed(2),5,285);\n [0,25,50,75,100].forEach(v=>{ctx.fillStyle=\'#9bb2c5\';ctx.fillText(v+\'%\',x(v)-12,378)});\n}\nfunction render(reset=false){\n const s=data.sessions[Number($(\'session\').value)],mode=$(\'mode\').value;\n if(reset){$(\'swing\').replaceChildren(new Option(\'All swings\',-1));s.ids.forEach((id,i)=>$(\'swing\').add(new Option(id,i)))}\n const highlight=Number($(\'swing\').value);\n const reviewed=(s.duration_review||[]).filter(r=>r.review_flags>0);\n $(\'duration-review\').innerHTML=reviewed.length?\'<p class="note">Duration review: \'+reviewed.map(r=>esc(title(r.phase))+\': \'+r.review_flags+\'/\'+r.n_swings+\' swings above \'+fmt(r.upper_fence_ms,1)+\' ms\').join(\'; \')+\'. These pooled Q3 + 3 IQR flags are review prompts. Primary curves retain every eligible swing. Screened comparisons are saved under phase_duration_audit.</p>\':\'<p class="small">No phase durations exceed the pooled Q3 + 3 IQR review fences. Fixed-cap sensitivity comparisons are saved separately.</p>\';\n\n $(\'counts\').innerHTML=\'<span class="pill">\'+s.ids.length+\' complete swings</span><span class="pill">Hitter \'+esc(s.key[0])+\'</span><span class="pill">Session \'+esc(s.key[1])+\' · \'+esc(s.key[2])+\'</span>\';\n $(\'timing\').innerHTML=table([\'Phase\',\'Mean ± SD (ms)\',\'Range (ms)\'],s.timing.map(r=>[title(r.phase),fmt(r.mean_duration_ms,1)+\' ± \'+fmt(r.sd_duration_ms,1),fmt(r.min_duration_ms,1)+\' – \'+fmt(r.max_duration_ms,1)]));\n $(\'plots\').replaceChildren();\n Object.entries(data.channels).forEach(([ch,label])=>data.phases.forEach(phase=>{\n  let raw=s.curves[phase+\'|\'+ch],m=raw.map(r=>r.map(v=>mode===\'start_relative\'?v-r[0]:v));let box=document.createElement(\'div\');box.className=\'panel\';\n  let h=document.createElement(\'h3\');h.textContent=label+\' · \'+title(phase);box.append(h);let cv=document.createElement(\'canvas\');cv.setAttribute(\'aria-label\',h.textContent+\' swing curves and standard deviation\');box.append(cv);$(\'plots\').append(box);plot(cv,m,ch.startsWith(\'pelvis\')?\'#58bbd0\':\'#f28da1\',highlight);\n }));\n let summary=s.summary.filter(r=>r.mode===mode);\n $(\'summary\').innerHTML=table([\'Channel\',\'Phase\',\'RMS trajectory SD (°)\',\'Peak SD (°)\',\'Peak phase progress\',\'Start SD (°)\',\'End SD (°)\'],summary.map(r=>[data.channels[r.channel],title(r.phase),fmt(r.rms_trajectory_sd_deg),fmt(r.peak_sd_deg),fmt(r.peak_sd_progress_pct,0)+\'%\',fmt(r.start_angle_sd_deg),fmt(r.end_angle_sd_deg)]));\n let dev=s.deviations.filter(r=>r.mode===mode).sort((a,b)=>b.loo_rms_distance_deg-a.loo_rms_distance_deg).slice(0,15);\n $(\'deviations\').innerHTML=table([\'Swing\',\'Channel\',\'Phase\',\'LOO RMS distance (°)\',\'Bat speed (mph)\',\'Exit velocity (mph)\'],dev.map(r=>[r.session_swing,data.channels[r.channel],title(r.phase),fmt(r.loo_rms_distance_deg),fmt(r.bat_speed_mph,1),fmt(r.exit_velocity_mph,1)]));\n let infl=s.influence.filter(r=>r.mode===mode).sort((a,b)=>Math.abs(b.change_rms_sd_deg)-Math.abs(a.change_rms_sd_deg)).slice(0,15);\n $(\'influence\').innerHTML=table([\'Omitted swing\',\'Channel\',\'Phase\',\'Baseline SD (°)\',\'Omission SD (°)\',\'Change (°)\',\'Remaining n\',\'Meets minimum\'],infl.map(r=>[r.omitted_swing,data.channels[r.channel],title(r.phase),fmt(r.baseline_rms_sd_deg),fmt(r.omission_rms_sd_deg),fmt(r.change_rms_sd_deg),r.remaining_n,r.remaining_meets_minimum?\'Yes\':\'No\']));\n}\n$(\'session\').addEventListener(\'change\',()=>render(true));$(\'mode\').addEventListener(\'change\',()=>render());$(\'swing\').addEventListener(\'change\',()=>render());render(true);\n</script></body></html>\n'
report_path = OUT / 'individual_swing_fingerprints.html'
report_path.write_text(REPORT_TEMPLATE.replace('__REPORT_DATA__', report_json), encoding='utf-8')
print('Interactive all-session report:', report_path)
display(FileLink(os.path.relpath(report_path, Path.cwd())))
try:
    import ipywidgets as widgets
    selection = widgets.Dropdown(options=[(' / '.join(k), k) for k in sorted(eligible_keys)], value=selected_key, description='Session:')
    mode = widgets.Checkbox(value=False, description='Start-relative angles')
    axial = widgets.Checkbox(value=False, description='Axial rotation')
    controls = widgets.HBox([selection, mode, axial])
    def preview(key, relative, rotation):
        plot_profile(key, relative, rotation, save=False)
    output = widgets.interactive_output(preview, {'key': selection, 'relative': mode, 'rotation': axial})
    display(controls, output)
except ImportError:
    print('Optional notebook widgets unavailable. Open the HTML report for session and swing selection.')

In [ ]:
for name, table in [('swing_features',features), ('session_counts',session_counts),
                    ('session_curve_metrics',metrics), ('session_timing_metrics',timing),
                    ('swing_leave_one_out_distances',deviations), ('swing_omission_influence',influence),
                    ('sensitivity_profiles',sensitivities), ('lookback_paired_changes',paired)]:
    table.to_csv(OUT / (name + '.csv'), index=False)
trajectory_rows = []
for trial, traces in curves.items():
    for (phase,ch), a in traces.items():
        trajectory_rows.extend({'session_swing':trial,'phase':phase,'channel':ch,'progress_pct':float(p*100),
                                'angle_deg':float(v),'start_relative_angle_deg':float(v-a[0])}
                               for p,v in zip(PROGRESS,a))
pd.DataFrame(trajectory_rows).to_csv(OUT / 'normalized_trajectories.csv.gz', index=False, compression='gzip')
input_names = ['joint_angles.csv','metadata.csv'] + (['poi_metrics.csv'] if optional_outcome_status == 'loaded' else [])
manifest = {'study':'16_individual_swing_fingerprints','dataset':'OpenBiomechanics Dataset v1',
            'lookback_ms':LOOKBACK_S*1000,'minimum_swings':MIN_VALID_SWINGS,'progress_points':len(PROGRESS),
            'channels':CHANNELS,'primary_filter':'published processed signals; no additional smoothing',
            'cohort':'complete three-phase swings; valid chronological events; separate hitter/session/side',
            'variance_band':'across-swing sample SD, ddof=1; not a confidence interval',
            'trajectory_sd':'sqrt integral of pointwise sample variance over normalized phase progress',
            'relative_reference':'each swing phase start; independent baseline per phase',
            'branch_alignment':'per-trial unwrap; whole-curve 360-degree shifts align phase starts within session',
            'loo_reference':'mean of all other retained session swings in the same channel, phase and mode',
            'omission_diagnostic':'remaining cohort may fall below reporting minimum; exported explicitly',
            'optional_outcomes_status':optional_outcome_status,
            'complete_swings':len(features),'eligible_profiles':len(eligible_keys),
            'input_sha256':{name:file_hash(RAW/name) for name in input_names},
            'phase_duration_audit':duration_audit_manifest,
            'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
interpretation = f"""Individual Swing Consistency Profiles

Retained {len(features)} complete swings and {len(eligible_keys)} hitter-session-side profiles at a minimum of {MIN_VALID_SWINGS} swings.
All primary profiles use the same complete swings across approach, landing, post-plant, and four angle channels.
Approach is {LOOKBACK_S*1000:g} ms before foot contact, not measured stride onset. Landing ends at full foot plant; post-plant ends at ball contact.
Absolute curves retain posture offsets. Start-relative curves remove each swing's starting angle separately within each phase.
Mean ±1 SD shows observed across-swing spread. The lower plot shows pointwise SD. These are not confidence intervals or clinical reference ranges.
Peak SD is a descriptive sample maximum, not evidence of a statistically localized unstable phase.
Timing variation is reported separately because normalized phases hide duration differences.
LOO distances and omission changes identify reps to review, not errors to delete automatically. Omission analyses can have one fewer swing than the reporting minimum.
Four-, five-, and six-swing thresholds change eligibility, not the measurement formula. Lookback comparisons use common trials and re-evaluate session eligibility.
Profiles with more swings have more stable estimates; no longitudinal/test-retest reliability or persistent biometric identity is established.
Bat speed and exit velocity are optional context only; no performance association or causal coaching recommendation is fitted.
Do not infer iPhone accuracy from motion capture: 2D posture proxies, camera view, occlusion, event annotation and repeated-session agreement require separate validation. Single-view CV does not recover these 3D axial angles directly.
Phase-duration audit flags pooled Q3 + 3*IQR durations without altering annotations or primary profiles. Fixed landing caps of 200/300/400 ms and a post-plant cap of 200 ms are analytical sensitivity settings, not validated physiological bounds.
Each scenario omits the same trial IDs from all phases and channels, recomputes profiles with the original minimum, and exports lost-session eligibility separately from paired changes. Paired comparisons therefore describe surviving sessions only.
See phase_duration_audit/ for trial flags, cohort counts, eligibility attrition, screened profiles, paired changes, duration distributions, and the real-time focused-session plot.
Raw inputs remain local. Official data license: https://github.com/drivelineresearch/openbiomechanics/blob/main/LICENSE-DATA.md
"""
(OUT/'interpretation.txt').write_text(interpretation,encoding='utf-8')
print(interpretation)
print('CSV summaries, audit, compressed curves, manifest and HTML report:', OUT)
print('Selected-session plots:', FIG)